# LlamaIndex Chat Engine 聊天引擎完全指南

## 📚 學習目標

本教程將深入講解：
1. Chat Engine 的核心概念
2. 多種聊天模式的使用
3. 對話歷史和記憶管理
4. 流式響應實作
5. 構建完整的智能客服系統

**難度**: ⭐⭐ 進階  
**預計時間**: 1 小時

## 🎯 Chat Engine vs Query Engine

| 特性 | Query Engine | Chat Engine |
|------|-------------|-------------|
| 上下文 | 無狀態，每次獨立查詢 | 有狀態，保留對話歷史 |
| 用途 | 單次問答 | 多輪對話 |
| 記憶 | 無 | 有 |
| 適用場景 | 文檔檢索、一次性查詢 | 聊天機器人、客服系統 |

## 環境設置

In [ ]:
# 安裝必要的套件
!pip install llama-index llama-index-llms-openai llama-index-embeddings-openai -q

In [ ]:
import os
from dotenv import load_dotenv

# 加載環境變數
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

print("✅ 環境設置完成")

## 準備測試數據和索引

In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Document

# 創建測試數據
os.makedirs("chatbot_data", exist_ok=True)

knowledge_base = {
    "product_info.txt": """我們的產品線包括：
    1. 智能音箱 - 售價 $99，支援語音控制和智能家居整合
    2. 智能手錶 - 售價 $299，具有健康監測和運動追蹤功能
    3. 無線耳機 - 售價 $149，主動降噪和長達24小時續航
    
    所有產品提供 1 年保固和 30 天退貨保證。""",
    
    "shipping_policy.txt": """運送政策：
    - 台灣本島：2-3 個工作天，免運費（訂單滿 $500）
    - 離島地區：3-5 個工作天，運費 $100
    - 國際運送：7-14 個工作天，運費依地區計算
    
    我們使用可追蹤的配送服務，訂單出貨後會收到追蹤號碼。""",
    
    "return_policy.txt": """退貨政策：
    - 購買後 30 天內可退貨
    - 商品需保持全新未使用狀態
    - 需附上原包裝和所有配件
    - 退款將在收到商品後 5-7 個工作天內處理
    
    聯絡客服請寄信至 support@example.com 或撥打 0800-123-456"""
}

for filename, content in knowledge_base.items():
    with open(f"chatbot_data/{filename}", "w", encoding="utf-8") as f:
        f.write(content)

# 加載文檔並創建索引
documents = SimpleDirectoryReader("chatbot_data").load_data()
index = VectorStoreIndex.from_documents(documents)

print(f"✅ 已創建索引，包含 {len(documents)} 個文檔")

## 1. SimpleChatEngine - 最基礎的聊天引擎

SimpleChatEngine 保留對話歷史，但不進行文檔檢索。

In [ ]:
from llama_index.core.chat_engine import SimpleChatEngine

# 創建簡單聊天引擎
simple_chat_engine = SimpleChatEngine.from_defaults()

print("💬 SimpleChatEngine 示例\n" + "="*50)

# 對話 1
response1 = simple_chat_engine.chat("你好，我叫 Alice")
print(f"User: 你好，我叫 Alice")
print(f"Assistant: {response1}\n")

# 對話 2 - 會記得上一輪對話
response2 = simple_chat_engine.chat("你還記得我的名字嗎？")
print(f"User: 你還記得我的名字嗎？")
print(f"Assistant: {response2}\n")

# 查看對話歷史
print("\n📜 對話歷史：")
for msg in simple_chat_engine.chat_history:
    print(f"{msg.role}: {msg.content}")

## 2. ContextChatEngine - 帶檢索的聊天引擎

這是最常用的聊天引擎，結合了文檔檢索和對話歷史。

In [ ]:
from llama_index.core.memory import ChatMemoryBuffer

# 創建記憶緩衝區
memory = ChatMemoryBuffer.from_defaults(token_limit=3000)

# 創建聊天引擎
chat_engine = index.as_chat_engine(
    chat_mode="context",  # 使用上下文模式
    memory=memory,
    system_prompt=(
        "你是一個友善的客服助手，專門回答關於產品、運送和退貨的問題。"
        "請基於提供的文檔信息回答，如果不確定請誠實告知。"
    )
)

print("💬 ContextChatEngine 示例\n" + "="*50 + "\n")

# 模擬客服對話
conversations = [
    "你好，請問有哪些產品？",
    "智能手錶多少錢？",
    "它有什麼功能？",
    "如果我要退貨怎麼辦？"
]

for question in conversations:
    response = chat_engine.chat(question)
    print(f"👤 User: {question}")
    print(f"🤖 Assistant: {response}\n")
    print("-" * 80 + "\n")

## 3. CondenseQuestionChatEngine - 壓縮歷史

將對話歷史壓縮成單一查詢，節省 token 和提高效率。

In [ ]:
# 創建 CondenseQuestion 模式的聊天引擎
condense_chat_engine = index.as_chat_engine(
    chat_mode="condense_question",
    verbose=True  # 顯示內部處理過程
)

print("💬 CondenseQuestionChatEngine 示例\n" + "="*50 + "\n")

# 對話 1
response1 = condense_chat_engine.chat("你們的運送政策是什麼？")
print(f"User: 你們的運送政策是什麼？")
print(f"Assistant: {response1}\n")

# 對話 2 - 使用代詞，需要理解上下文
response2 = condense_chat_engine.chat("那離島呢？")
print(f"User: 那離島呢？")
print(f"Assistant: {response2}\n")

## 4. ReAct Agent Chat - 最強大的模式

ReAct Agent 可以使用工具並進行推理。

In [ ]:
from llama_index.core.tools import QueryEngineTool

# 為不同的文檔創建專門的查詢引擎
product_index = VectorStoreIndex.from_documents(
    [doc for doc in documents if "product" in doc.metadata.get("file_name", "")]
)
shipping_index = VectorStoreIndex.from_documents(
    [doc for doc in documents if "shipping" in doc.metadata.get("file_name", "")]
)

# 創建工具
product_tool = QueryEngineTool.from_defaults(
    query_engine=product_index.as_query_engine(),
    description="用於查詢產品信息、價格和規格"
)

shipping_tool = QueryEngineTool.from_defaults(
    query_engine=shipping_index.as_query_engine(),
    description="用於查詢運送政策、時間和費用"
)

# 創建 ReAct Agent 聊天引擎
agent_chat_engine = index.as_chat_engine(
    chat_mode="react",
    tools=[product_tool, shipping_tool],
    verbose=True
)

print("💬 ReAct Agent Chat 示例\n" + "="*50 + "\n")

# Agent 會自動選擇合適的工具
response = agent_chat_engine.chat(
    "我想買智能手錶，請告訴我價格，以及如果我在離島，運送需要多久？"
)
print(f"\n🤖 Final Answer: {response}")

## 5. 流式響應

提供打字機效果，改善用戶體驗。

In [ ]:
# 創建支援流式輸出的聊天引擎
streaming_chat_engine = index.as_chat_engine(
    chat_mode="context",
    streaming=True
)

print("💬 流式響應示例\n" + "="*50 + "\n")

question = "請詳細介紹一下你們的產品"
print(f"User: {question}")
print(f"Assistant: ", end="")

# 流式輸出
streaming_response = streaming_chat_engine.stream_chat(question)

for token in streaming_response.response_gen:
    print(token, end="", flush=True)

print("\n")

## 6. 對話歷史管理

In [ ]:
from llama_index.core.llms import ChatMessage, MessageRole

# 創建聊天引擎
chat_engine = index.as_chat_engine(chat_mode="context")

# 手動添加對話歷史
chat_history = [
    ChatMessage(role=MessageRole.USER, content="你好"),
    ChatMessage(role=MessageRole.ASSISTANT, content="你好！有什麼可以幫助你的嗎？"),
    ChatMessage(role=MessageRole.USER, content="我想了解智能手錶"),
]

# 設置歷史
chat_engine.chat_history = chat_history

# 繼續對話
response = chat_engine.chat("它有哪些功能？")
print(f"Response: {response}")

# 重置對話
chat_engine.reset()
print("\n✅ 對話已重置")

## 7. 完整的智能客服系統

結合所有功能，構建一個生產級的客服系統。

In [ ]:
from typing import List, Dict
from datetime import datetime

class SmartCustomerService:
    """智能客服系統"""
    
    def __init__(self, index, system_prompt: str = None):
        self.index = index
        self.sessions = {}  # 儲存多個用戶的會話
        
        self.default_prompt = system_prompt or (
            "你是一個專業、友善的客服助手。"
            "請基於知識庫回答用戶問題。"
            "如果不確定，請誠實告知並建議聯絡人工客服。"
        )
    
    def create_session(self, user_id: str):
        """為用戶創建新會話"""
        chat_engine = self.index.as_chat_engine(
            chat_mode="context",
            system_prompt=self.default_prompt
        )
        
        self.sessions[user_id] = {
            "engine": chat_engine,
            "created_at": datetime.now(),
            "message_count": 0
        }
        
        return f"✅ 會話已創建，用戶 ID: {user_id}"
    
    def chat(self, user_id: str, message: str) -> str:
        """處理用戶消息"""
        # 如果會話不存在，創建新會話
        if user_id not in self.sessions:
            self.create_session(user_id)
        
        session = self.sessions[user_id]
        chat_engine = session["engine"]
        
        # 獲取回應
        response = chat_engine.chat(message)
        
        # 更新統計
        session["message_count"] += 1
        
        return str(response)
    
    def get_history(self, user_id: str) -> List[Dict]:
        """獲取用戶的對話歷史"""
        if user_id not in self.sessions:
            return []
        
        engine = self.sessions[user_id]["engine"]
        return [
            {"role": msg.role.value, "content": msg.content}
            for msg in engine.chat_history
        ]
    
    def reset_session(self, user_id: str):
        """重置用戶會話"""
        if user_id in self.sessions:
            self.sessions[user_id]["engine"].reset()
            self.sessions[user_id]["message_count"] = 0
            return f"✅ 用戶 {user_id} 的會話已重置"
        return f"❌ 用戶 {user_id} 沒有活躍會話"
    
    def get_stats(self) -> Dict:
        """獲取系統統計"""
        return {
            "total_sessions": len(self.sessions),
            "active_users": list(self.sessions.keys()),
            "total_messages": sum(
                s["message_count"] for s in self.sessions.values()
            )
        }

# 使用示例
print("🤖 智能客服系統啟動\n" + "="*80 + "\n")

# 創建客服系統
customer_service = SmartCustomerService(index)

# 模擬多個用戶
users = ["user_001", "user_002"]

# 用戶 1 的對話
print("👤 用戶 user_001 進入對話\n")
print(customer_service.create_session("user_001"))
response1 = customer_service.chat("user_001", "你好，我想了解智能手錶")
print(f"🤖: {response1}\n")

response2 = customer_service.chat("user_001", "價格是多少？")
print(f"🤖: {response2}\n")

# 用戶 2 的對話
print("\n" + "-"*80 + "\n")
print("👤 用戶 user_002 進入對話\n")
response3 = customer_service.chat("user_002", "運送到離島要多久？")
print(f"🤖: {response3}\n")

# 查看統計
print("\n" + "="*80)
print("📊 系統統計：")
stats = customer_service.get_stats()
for key, value in stats.items():
    print(f"  {key}: {value}")

# 查看用戶歷史
print("\n📜 用戶 user_001 的對話歷史：")
history = customer_service.get_history("user_001")
for msg in history:
    print(f"  {msg['role']}: {msg['content']}")

## 8. 互動式聊天界面

In [ ]:
def interactive_chat(index):
    """互動式聊天界面"""
    chat_engine = index.as_chat_engine(
        chat_mode="context",
        system_prompt="你是一個友善的客服助手。"
    )
    
    print("\n" + "="*80)
    print("💬 歡迎使用智能客服系統")
    print("輸入 'exit' 或 'quit' 退出")
    print("輸入 'reset' 重置對話")
    print("輸入 'history' 查看對話歷史")
    print("="*80 + "\n")
    
    while True:
        user_input = input("👤 You: ")
        
        if user_input.lower() in ['exit', 'quit', '退出']:
            print("\n👋 感謝使用，再見！")
            break
        
        if user_input.lower() == 'reset':
            chat_engine.reset()
            print("✅ 對話已重置\n")
            continue
        
        if user_input.lower() == 'history':
            print("\n📜 對話歷史：")
            for msg in chat_engine.chat_history:
                print(f"  {msg.role.value}: {msg.content}")
            print()
            continue
        
        if not user_input.strip():
            continue
        
        try:
            response = chat_engine.chat(user_input)
            print(f"🤖 Assistant: {response}\n")
        except Exception as e:
            print(f"❌ 錯誤: {e}\n")

# 取消下面的註釋來啟動互動式聊天
# interactive_chat(index)

print("💡 提示：取消上面的註釋來啟動互動式聊天界面")

## 📝 本教程總結

### ✅ 已掌握的技能

1. **Chat Engine 類型**:
   - ✅ SimpleChatEngine - 基礎對話
   - ✅ ContextChatEngine - 帶檢索的對話（最常用）
   - ✅ CondenseQuestionChatEngine - 壓縮歷史
   - ✅ ReAct Agent - 工具使用和推理

2. **高級功能**:
   - ✅ 對話歷史管理
   - ✅ 流式響應
   - ✅ 會話狀態管理
   - ✅ 自定義系統提示詞

3. **實戰應用**:
   - ✅ 智能客服系統
   - ✅ 多用戶會話管理
   - ✅ 對話統計和監控

### 🎯 Chat Mode 選擇指南

| Chat Mode | 使用場景 | 優點 | 缺點 |
|-----------|---------|------|------|
| **simple** | 簡單對話，無需檢索 | 快速、低成本 | 無知識庫 |
| **context** | 一般客服、問答 | 平衡性能和功能 | Token 消耗較多 |
| **condense_question** | 長對話 | 節省 Token | 可能丟失上下文 |
| **react** | 複雜任務、需要工具 | 最強大、最靈活 | 速度較慢、成本高 |

### 💡 生產環境最佳實踐

1. **記憶管理**:
   - 設置合理的 token_limit（3000-4000）
   - 定期清理舊會話
   - 實現會話持久化

2. **性能優化**:
   - 使用流式響應改善體驗
   - 快取常見問題的答案
   - 異步處理提高並發

3. **用戶體驗**:
   - 清晰的系統提示詞
   - 適時的澄清問題
   - 提供人工客服轉接

4. **監控與評估**:
   - 記錄對話日誌
   - 追蹤用戶滿意度
   - 分析常見問題

### 🚀 下一步

- **5.Agents與工具使用.ipynb**: 深入學習 Agent 系統
- **6.多模態RAG.ipynb**: 處理圖片和多媒體
- **案例1_企業文檔問答系統.ipynb**: 完整實戰項目

### 🔗 延伸閱讀

- [Chat Engine 官方文檔](https://docs.llamaindex.ai/en/stable/module_guides/deploying/chat_engines/)
- [Memory 管理](https://docs.llamaindex.ai/en/stable/module_guides/deploying/chat_engines/usage_pattern.html)
- [Agent 系統](https://docs.llamaindex.ai/en/stable/module_guides/deploying/agents/)